# A3 — M2 runner: unimodal baselines + simple fusion (Colab)

This notebook holds **no model logic**. It prepares Colab and calls the scripts in `Assignment3/code/`:

| Step | Script | Cost (A100 / T4) |
|---|---|---|
| 1. Frozen features (RoBERTa, ViT-B/16 ImageNet), run once, cached on Drive | `extract_features.py` | ~10–20 / 20–40 min total |
| 2. Text-only and image-only heads, 3 seeds, text pooling (CLS / mean) chosen on **val** macro-F1 | `train.py` | seconds per run |
| 3. Concatenation fusion with the selected text + image features, 3 seeds | `train.py` | seconds per run |
| 4. Tables, per-class F1, confusion matrices, support/conflict counts, checkpoints | `evaluate.py` | seconds |

**Before running:** push the latest `code/` to GitHub (Colab clones the repo) and pick *Runtime → Change runtime type → A100 GPU* (T4 also works).
Image decoding on the CPU is the bottleneck of feature extraction, so the notebook sizes the DataLoader workers from the CPU count of the runtime.
Everything that must survive a disconnect (features, runs, results) is written to Google Drive under `DL_A3/`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 0. Configuration

In [ ]:
from pathlib import Path

REPOSITORY_URL = "https://github.com/MinhTrinhh/deep-learning-landing-page.git"
BRANCH = "main"
SEEDS = [0, 1, 2]

TEXT_FEATS = ["roberta_cls", "roberta_mean"]   # proposal: CLS; mean pooling is the usual stronger frozen variant
IMAGE_FEATS = ["vit"]                          # ImageNet ViT-B/16

DRIVE_ROOT = Path("/content/drive/MyDrive/DL_A3")
FEATURES_DIR = DRIVE_ROOT / "features"
RUNS_DIR = DRIVE_ROOT / "runs"
RESULTS_DRIVE = DRIVE_ROOT / "m2_results"

DATA_DIR = Path("/content/n24news")            # local disk: much faster than reading 61k images from Drive
REPO_DIR = Path("/content/deep-learning-landing-page")
A3_DIR = REPO_DIR / "Assignment3"
GDRIVE_ID = "1OS1fXwZ1Vsj70lEQajccyssxQRYp5X9D"  # N24News.zip from the official README
SAVE_ZIP_TO_DRIVE = False                      # True keeps a copy of the zip on Drive (no re-download next session)

import os
NUM_WORKERS = max(2, min(8, (os.cpu_count() or 2) - 1))   # JPEG decoding is CPU-bound
print("CPUs:", os.cpu_count(), "-> DataLoader workers:", NUM_WORKERS)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
for d in (FEATURES_DIR, RUNS_DIR, RESULTS_DRIVE):
    d.mkdir(parents=True, exist_ok=True)

## 1. Code: clone or update the repository

In [ ]:
import subprocess

if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPOSITORY_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "log", "-1", "--oneline"], check=True)
print(sorted(p.name for p in (A3_DIR / "code").iterdir()))

In [ ]:
# Colab already ships torch, transformers, scikit-learn, pandas and matplotlib; gdown is only for the download
!pip -q install gdown
import torch, transformers, sklearn
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| sklearn", sklearn.__version__, "| cuda", torch.cuda.is_available())
GPU_NAME = torch.cuda.get_device_name() if torch.cuda.is_available() else "cpu"
GPU_MEM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
EXTRACT_BATCH = 512 if GPU_MEM_GB >= 30 else 128   # A100 40/80 GB vs T4 16 GB
print(f"{GPU_NAME} ({GPU_MEM_GB:.0f} GB) -> extraction batch size {EXTRACT_BATCH}")

In [ ]:
import sys

def run(*args):
    """Run a script from Assignment3/ with live output; raise (stop the notebook) if it fails."""
    cmd = [sys.executable, "-u", *map(str, args)]
    print("$", " ".join(cmd[1:]), flush=True)
    proc = subprocess.Popen(cmd, cwd=A3_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    while chunk := proc.stdout.read1(4096):
        sys.stdout.write(chunk.decode(errors="replace"))
        sys.stdout.flush()
    if proc.wait() != 0:
        raise RuntimeError(f"command failed with exit code {proc.returncode}")

## 2. Data: N24News on the local disk

In [ ]:
import shutil

zip_local = DATA_DIR / "N24News.zip"
zip_drive = DRIVE_ROOT / "N24News.zip"
DATA_DIR.mkdir(parents=True, exist_ok=True)

if not any(DATA_DIR.rglob("nytimes.json")):
    if not zip_local.exists():
        if zip_drive.exists():
            shutil.copy(zip_drive, zip_local)
        else:
            # If gdown fails with a quota error: open https://drive.google.com/file/d/1OS1fXwZ1Vsj70lEQajccyssxQRYp5X9D,
            # add it to your Drive, put N24News.zip in MyDrive/DL_A3/ and rerun this cell.
            import gdown
            gdown.download(id=GDRIVE_ID, output=str(zip_local), quiet=False)
            if SAVE_ZIP_TO_DRIVE:
                shutil.copy(zip_local, zip_drive)
    subprocess.run(["unzip", "-q", "-n", str(zip_local), "-d", str(DATA_DIR)], check=True)

print("metadata:", next(DATA_DIR.rglob("nytimes.json")))
print("images:", sum(1 for _ in DATA_DIR.rglob("*.jpg")))

## 3. Frozen features (once)

Each encoder writes `<name>.npy` (float16, one row per line of `outputs/eda/splits.csv`) to Drive and is skipped on later sessions.
Text: headline + abstract as a RoBERTa sentence pair, max 128 tokens. Images: shorter side → 224, centre crop 224 (drops the year badge), encoder normalization.

In [ ]:
for encoder in ["roberta", "vit"]:
    run("code/extract_features.py", "--data", DATA_DIR, "--out", FEATURES_DIR, "--encoder", encoder,
        "--batch-size", EXTRACT_BATCH, "--num-workers", NUM_WORKERS)
!ls -lh {FEATURES_DIR}

## 4. Unimodal baselines

Every feature variant × 3 seeds. The checkpoint is selected on validation macro-F1; the test split is scored once per run.

In [ ]:
for feat in TEXT_FEATS:
    for seed in SEEDS:
        run("code/train.py", "--features", FEATURES_DIR, "--out", RUNS_DIR, "--model", "text",
            "--text-feat", feat, "--seed", seed)

for feat in IMAGE_FEATS:
    for seed in SEEDS:
        run("code/train.py", "--features", FEATURES_DIR, "--out", RUNS_DIR, "--model", "image",
            "--image-feat", feat, "--seed", seed)

## 5. Simple fusion with the features selected on validation

In [ ]:
def best(kind):
    args = [sys.executable, "code/evaluate.py", "--runs", str(RUNS_DIR), "--best", kind]
    out = subprocess.run(args, cwd=A3_DIR, capture_output=True, text=True, check=True).stdout
    return out.strip().splitlines()[-1]

BEST_TEXT, BEST_IMAGE = best("text"), best("image")
print("selected on val macro-F1 -> text:", BEST_TEXT, "| image:", BEST_IMAGE)

for seed in SEEDS:
    run("code/train.py", "--features", FEATURES_DIR, "--out", RUNS_DIR, "--model", "concat",
        "--text-feat", BEST_TEXT, "--image-feat", BEST_IMAGE, "--seed", seed)

## 6. Evaluation: tables, figures, checkpoints

In [ ]:
RESULTS_DIR = A3_DIR / "outputs" / "metrics" / "m2"
eval_args = ["code/evaluate.py", "--runs", RUNS_DIR, "--out", RESULTS_DIR,
             "--checkpoints", A3_DIR / "checkpoints", "--data", DATA_DIR]
run(*eval_args)

shutil.copytree(RESULTS_DIR, RESULTS_DRIVE, dirs_exist_ok=True)
for ckpt in (A3_DIR / "checkpoints").glob("*.pt"):
    shutil.copy(ckpt, RESULTS_DRIVE / ckpt.name)
print("copied to", RESULTS_DRIVE)

In [ ]:
import pandas as pd
from IPython.display import Image, Markdown, display

display(Markdown((RESULTS_DIR / "results.md").read_text()))
display(pd.read_csv(RESULTS_DIR / "modality_agreement.csv"))
display(pd.read_csv(RESULTS_DIR / "per_class_f1.csv", index_col=0).style.format("{:.3f}").background_gradient(axis=None))
for kind in ["text", "image", "concat"]:
    display(Image(str(RESULTS_DIR / "figures" / f"confusion_{kind}.png"), width=700))

## 7. Bring the results back to the repo

Download the zip, unzip it into `Assignment3/` on the laptop and commit `outputs/metrics/m2/` (tables + figures) and `checkpoints/*.pt` (small heads, < 4 MB each).
The feature files and per-run folders stay on Drive.

In [ ]:
import zipfile
from google.colab import files

archive = Path("/content/a3_m2_results.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for path in list(RESULTS_DIR.rglob("*")) + list((A3_DIR / "checkpoints").glob("*.pt")):
        if path.is_file():
            z.write(path, path.relative_to(A3_DIR))
print(archive, f"{archive.stat().st_size / 1e6:.1f} MB")
files.download(str(archive))